In [117]:
# Model takes a list of sentences and outputs an array of score with a formality score fore each sentence.




In [118]:
# Imports
import numpy as np

# The word2vec imports

import gensim
import gensim.downloader as api
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# The LSTM imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn import TransformerEncoder, TransformerEncoderLayer

In [174]:
class BidirectionalTransformer(nn.Module):
    def __init__(self, input_dim, model_dim, num_heads, num_layers, dropout=0.1):
        super(BidirectionalTransformer, self).__init__()
        
        # Positional encoding (optional if you're working with sequences)
        self.positional_encoding = nn.Embedding(5000, model_dim)  # You can customize the maximum length
        
        # Transformer encoder layers
        encoder_layer = TransformerEncoderLayer(d_model=model_dim, nhead=num_heads, dropout=dropout)
        self.transformer_encoder = TransformerEncoder(encoder_layer, num_layers=num_layers)
    
        
    def forward(self, x):
        # Optionally add positional encoding (for sequences)
        seq_len, batch_size = x.size(0), x.size(1)
        positions = torch.arange(0, seq_len).unsqueeze(1).expand(seq_len, batch_size).to(x.device)
        x = x + self.positional_encoding(positions)
        
        # Transformer Encoder (bidirectional)
        encoded_output = self.transformer_encoder(x)
        
       
        
        return encoded_output
    
class FullyConnectedLayer(nn.Module):
    def __init__(self, input_size, output_size=1):
        super(FullyConnectedLayer, self).__init__()
        self.fc = nn.Linear(input_size, output_size)
    
    def forward(self, x):
        # Ensure the input is 1D
        if x.ndim != 1:
            raise ValueError("Input tensor must be 1D")
        
        # Check if the input size matches the expected input size (625 in this case)
        if x.size(0) != 625:  # Assuming input_size is 625
            raise ValueError("Input tensor must have length 625")

        output = self.fc(x)  # Apply the fully connected layer
        output = torch.sigmoid(output)  # Apply sigmoid activation
        return output


    

def sentences_to_vectors(sentences, model):
    encoding_dim = model.vector_size
    no_sentence = len(sentences)
    max_word_count = 25
    
    # Initialize a 3D array with zeros
    returned_array = np.zeros((encoding_dim, max_word_count, no_sentence))

    def tokenize(sentence):
        tokens = word_tokenize(sentence)  # Tokenize sentence
        return [word for word in tokens if word not in stopwords.words('english')]  # Remove stopwords

    for i, sentence in enumerate(sentences):
        words = tokenize(sentence)
        word_vectors = [model[word] for word in words if word in model]
        
        # Pad or truncate word_vectors to fit max_word_count
        if len(word_vectors) < max_word_count:
            # Pad with zeros if there are fewer than max_word_count word vectors
            padded_vectors = np.array(word_vectors + [[0] * encoding_dim] * (max_word_count - len(word_vectors)))
        else:
            # Truncate if there are more than max_word_count word vectors
            padded_vectors = np.array(word_vectors[:max_word_count])
        
        # Fill the 3D array
        returned_array[:, :, i] = padded_vectors.T  # Transpose to match shape (encoding_dim, max_word_count)

    return returned_array

In [120]:
print(list(api.info()['models'].keys()))

['fasttext-wiki-news-subwords-300', 'conceptnet-numberbatch-17-06-300', 'word2vec-ruscorpora-300', 'word2vec-google-news-300', 'glove-wiki-gigaword-50', 'glove-wiki-gigaword-100', 'glove-wiki-gigaword-200', 'glove-wiki-gigaword-300', 'glove-twitter-25', 'glove-twitter-50', 'glove-twitter-100', 'glove-twitter-200', '__testing_word2vec-matrix-synopsis']


In [121]:
model = api.load('glove-twitter-25')

In [122]:
example_preprocessed_sentences = [
    "team wanted provide update latest progress marketing campaign",
    "successfully completed initial phase client happy results far",
    "however changes need addressed move forward",
    "arrange meeting next week go revised strategy ensure page",
    "please let know availability",
    "looking forward continued collaboration"
]

example_targets = np.array([0,1,1,1,0,0])
targets_tensor = torch.from_numpy(example_targets).float()


In [123]:
sentence_vectors = sentences_to_vectors(example_preprocessed_sentences, model)
print(sentence_vectors.shape)

(25, 25, 6)


In [124]:
# LSTM hyperparameters 

input_dim = 25       # Example input dimension (e.g., number of features)
model_dim = 25       # Dimension of the model (embedding size)
num_heads = 5       # Number of attention heads
num_layers = 1       # Number of transformer layers
dropout = 0.2        # Dropout rate

fc_dim = 625


# Training hyperparameters 

sequence_length = 1  # Each sample is a single time step
batch_size = sentence_vectors.shape[1]  # All samples in one batch
input_size = sentence_vectors.shape[0]  # Number of features

In [176]:
lstm = BidirectionalTransformer(input_dim, model_dim, num_heads, num_layers, dropout)
fc_layer =  FullyConnectedLayer(fc_dim)

In [126]:
# Training Hyperparameters

num_epochs = 20       # Number of training epochs

# Optimizer Hyperparameters

opt_lr = 0.001

criterion = nn.BCELoss()

In [152]:
criterion(1,2)

AttributeError: 'int' object has no attribute 'size'

In [184]:
# The training process!
lstm.train()
criterion = nn.BCELoss()
optimizer = optim.Adam(lstm.parameters(), lr=opt_lr)

for epoch in range(num_epochs):
    optimizer.zero_grad()  # Clear previous gradients
    
    # Initialize total_epoch_loss as a float
    total_epoch_loss = torch.tensor(0.0, requires_grad=True)

    for sen_ix in range(sentence_vectors.shape[2]):
        sentence_array = sentence_vectors[:, :, sen_ix]
        input_tensor = torch.from_numpy(sentence_array).float().view(sequence_length, batch_size, input_size)
        output_tensor = lstm(input_tensor)  # Get output from the transformer
        
        # Use the last time step's output directly
        output = fc_layer(torch.flatten(output_tensor))  # Use the last time step’s output
        
        # Flatten output if needed
        output = output.view(-1)  
        

        # Ensure targets_tensor is properly shaped
        target_tensor = torch.tensor(targets_tensor[sen_ix], dtype=torch.float32).unsqueeze(0)  # Convert target to tensor
        
        
        # Compute loss
        sentence_loss = criterion(output, target_tensor)  
        
        # Accumulate loss
        total_epoch_loss = total_epoch_loss + sentence_loss

    print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {total_epoch_loss:.4f}")
    total_epoch_loss.backward()  # Backpropagate
    optimizer.step()  # Update parameters


C:\Users\timur\AppData\Local\Temp\ipykernel_19616\1602174309.py:25: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target_tensor = torch.tensor(targets_tensor[sen_ix], dtype=torch.float32).unsqueeze(0)  # Convert target to tensor


Epoch [1/20], Loss: 0.4252
Epoch [2/20], Loss: 0.4833
Epoch [3/20], Loss: 0.5105
Epoch [4/20], Loss: 0.4102
Epoch [5/20], Loss: 0.3773
Epoch [6/20], Loss: 0.3130
Epoch [7/20], Loss: 0.3133
Epoch [8/20], Loss: 0.3332
Epoch [9/20], Loss: 0.2720
Epoch [10/20], Loss: 0.3293
Epoch [11/20], Loss: 0.4135
Epoch [12/20], Loss: 0.2842
Epoch [13/20], Loss: 0.3557
Epoch [14/20], Loss: 0.2861
Epoch [15/20], Loss: 0.2957
Epoch [16/20], Loss: 0.2722
Epoch [17/20], Loss: 0.3224
Epoch [18/20], Loss: 0.2493
Epoch [19/20], Loss: 0.2453
Epoch [20/20], Loss: 0.2186


In [42]:
total_output_array = np.zeros_like(sentence_vectors)
for sen_ix in range(sentence_vectors.shape[2]):


    sentence_array = sentence_vectors[:,:,sen_ix]
    input_tensor = torch.from_numpy(sentence_array).float().view(sequence_length, batch_size, input_size)
    output_tensor = lstm(input_tensor)
    output_array = output_tensor.detach().numpy()

    total_output_array[:,:,sen_ix] = output_array
    
  


In [43]:
total_output_array

array([[[-3.80215317e-01, -4.97545153e-01, -4.43819880e-01,
         -6.91345990e-01, -4.54971462e-01, -7.13925362e-01],
        [ 7.37200499e-01,  3.35420668e-01,  4.13191915e-01,
          7.35168159e-01,  7.10988283e-01,  6.43464804e-01],
        [-8.18870187e-01, -1.31111658e+00, -7.57684231e-01,
         -3.74326229e-01, -1.13643932e+00, -1.02500355e+00],
        ...,
        [ 3.96366537e-01, -2.81659365e-02,  2.90656656e-01,
          3.75566363e-01,  6.45491898e-01, -1.47516891e-01],
        [ 9.46863890e-01,  9.88466382e-01,  8.72853518e-01,
          7.89522409e-01,  1.10992646e+00,  7.41994023e-01],
        [-3.52404714e-01, -1.56994477e-01, -5.82923234e-01,
         -2.26171955e-01, -6.09160602e-01, -3.16285968e-01]],

       [[-2.48455197e-01, -1.55059755e-01, -8.40715766e-02,
         -4.44564253e-01, -3.27482134e-01, -3.52930665e-01],
        [ 5.14856696e-01,  2.85680592e-01,  4.75950241e-01,
          3.13885659e-01,  2.76513517e-01,  4.68753576e-01],
        [-6.56107